# D250 — Olist on HDFS and Hive: Student Ingestion Exercise

Prepare the nine Olist CSV files, place them in isolated HDFS directories, register Hive external tables, validate the data, and create a curated columnar layer. You are given the workflow and checkpoints, not the table DDL answers.


## Lab conventions

- Run this notebook from Jupyter in Ubuntu/WSL with HDFS, YARN, the Hive metastore, and HiveServer2 running.
- Source CSVs are expected at `/mnt/c/data/olist`. Change `OLIST_SOURCE` if your copy is elsewhere.
- Use database `olist_hive`. Raw external tables should be named after the MySQL tables: `olist_customers`, `olist_orders`, `olist_order_items`, `olist_order_payments`, `olist_order_reviews`, `olist_products`, `olist_sellers`, `olist_geolocation`, and `product_category_translation`.
- These are learner exercises. TODO cells intentionally contain no completed answer.


## Learning objectives

- inspect CSV structure before ingestion;
- remove exactly the first/header line without changing the original file;
- upload one headerless CSV per HDFS table directory;
- define external Hive tables with appropriate SerDes and data types;
- distinguish an HDFS file from a Hive catalog table;
- validate row grain, nulls, keys, and joins; and
- create reusable Parquet or ORC tables with CTAS.


## 0. Check services


In [ ]:
%%bash
jps
ss -lnt | grep -E ':(9083|10000)\b' || true
hdfs dfsadmin -report | grep -E 'Live datanodes|Name:'
yarn node -list


## 1. Inspect the source directory

Do not upload the source files immediately. First list them, inspect the first two physical lines, record sizes, and confirm whether quoted commas or embedded line breaks occur. The review file deserves special attention because free text can be awkward for line-oriented CSV readers.


In [ ]:
%%bash
OLIST_SOURCE=/mnt/c/data/olist
ls -lh "$OLIST_SOURCE"/*.csv
for file in "$OLIST_SOURCE"/*.csv; do
  echo "=== $(basename "$file") ==="
  sed -n '1,2p' "$file"
done


## 2. Remove the first line, then place the files in HDFS

For **every CSV**, remove the header first and only then upload the headerless copy. Use:

```bash
sed '1d' SOURCE.csv > /tmp/HEADERLESS.csv
hdfs dfs -mkdir -p HDFS_TABLE_DIRECTORY
hdfs dfs -put -f /tmp/HEADERLESS.csv HDFS_TABLE_DIRECTORY/
```

`sed '1d'` removes line 1 from the output; it does not edit the original Windows file. Keep every dataset in its own HDFS directory so a Hive table cannot accidentally read another table's files. Do not use `tail -n +2` unless you can explain that it has the same purpose here.


In [ ]:
%%bash
# TODO: For all nine files, create headerless copies under /tmp/d250_olist/.
# Required source names:
# olist_customers_dataset.csv, olist_orders_dataset.csv,
# olist_order_items_dataset.csv, olist_order_payments_dataset.csv,
# olist_order_reviews_dataset.csv, olist_products_dataset.csv,
# olist_sellers_dataset.csv, olist_geolocation_dataset.csv,
# product_category_name_translation.csv
#
# Then place each file under:
# /user/hive/external/d250_olist/<table_name>/
#
# Keep any repeatable cleanup restricted to /user/hive/external/d250_olist.
# List the final HDFS tree and show the first row of each uploaded file.


### Checkpoint: expected logical row counts

The common Kaggle release has these counts after removing one header row:

| Dataset | Rows |
|---|---:|
| customers | 99,441 |
| orders | 99,441 |
| order items | 112,650 |
| payments | 103,886 |
| reviews | 99,224 |
| products | 32,951 |
| sellers | 3,095 |
| geolocation | 1,000,163 |
| category translation | 71 |

Physical line counts can disagree with logical CSV records when quoted text contains a newline. Investigate rather than forcing a count to match.


## 3. Create the Hive database and raw external tables

Create database `olist_hive`, then create one external table per HDFS directory. Requirements:

- represent every source column;
- preserve missing CSV values as null where practical;
- choose `STRING`, numeric, `DATE`, or `TIMESTAMP` types deliberately;
- use a CSV-aware SerDe where quoting or commas inside text require it;
- point each table at its exact HDFS directory;
- do not use `LOAD DATA` after creating location-based external tables; and
- document any limitation involving multiline review text.

Tip: `org.apache.hadoop.hive.serde2.OpenCSVSerde` understands quoted commas but commonly exposes fields as strings. A robust design may land raw columns as strings, then cast and clean them in a CTAS layer.


In [ ]:
%%bash
# TODO: Create the database and nine raw external tables. Add DESCRIBE FORMATTED checks.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## 4. Validate ingestion

Write validation queries that answer all of these:

1. How many rows does each table contain?
2. Are the expected identifiers null?
3. Are candidate keys unique at their stated grain?
4. What are the minimum and maximum order purchase timestamps?
5. Do numeric casts for price, freight, payment value, and review score succeed?
6. How many orders lack a matching customer?
7. How many order-item rows lack a matching order, product, or seller?

Tip: a row count is not always an order count. Use `COUNT(DISTINCT ...)` when the business entity is an order or customer.


In [ ]:
%%bash
# TODO: Add validation queries and briefly annotate any discrepancy.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## 5. Create a curated columnar layer

Create managed Parquet or ORC tables from the raw external tables. Cast timestamps and measures, turn empty strings into nulls, and retain the original table names only if your instructor confirms the naming convention. A safer option is a `cur_` prefix, for example `cur_orders`.

For the orders table, consider a derived `purchase_year` and `purchase_month`. Do not partition blindly: justify any partition key using expected filters and partition cardinality.


In [ ]:
%%bash
# TODO: Create curated CTAS tables and validate their schemas and row counts.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## MySQL → HiveSQL reminders

| MySQL habit | HiveSQL approach in this lab |
|---|---|
| `LIMIT offset, count` | use `LIMIT count`; use `ROW_NUMBER()` when paging is genuinely required |
| `IFNULL(x, y)` | prefer standard `COALESCE(x, y)` |
| `DATEDIFF(end, start)` | same two-argument order in Hive; result is whole days |
| `DATE_FORMAT(ts, '%Y-%m')` | use `date_format(ts, 'yyyy-MM')` |
| `YEAR(ts)` / `MONTH(ts)` | `year(ts)` / `month(ts)` are available |
| `GROUP_CONCAT` | use `concat_ws(',', collect_list(...))` when appropriate |
| backtick-heavy identifiers | use simple lowercase snake_case names; backticks are rarely needed here |
| primary keys, foreign keys, and row indexes | Hive generally does not enforce relational keys; performance comes from file format, partitions, statistics, pruning, and execution settings |
| `EXPLAIN ANALYZE` | use `EXPLAIN`, `EXPLAIN FORMATTED`, and runtime/Tez or YARN evidence available in your installation |

Hive aliases usually cannot be reused by another expression in the same `SELECT`. Put the first calculation in a CTE/subquery. Always make window ordering deterministic by adding an ID as a tie-breaker.


## 6. Student practice questions

1. Count orders by status and purchase month.
2. Calculate delivery duration and compare actual with estimated delivery dates.
3. Find repeat buyers using `customer_unique_id`, not the order-specific `customer_id`.
4. Compare item totals (`price + freight_value`) with payment totals without multiplying rows across two one-to-many tables.
5. Translate product categories and rank them by item revenue.
6. Compare review scores for on-time and late deliveries.
7. Reduce geolocation to one representative coordinate per ZIP prefix before joining it to customers or sellers.
8. Explain which objects live only in HDFS, which live in the Hive metastore, and which store managed data.

For every result, state what one output row represents and how nulls and duplicate business entities were handled.


In [ ]:
%%bash
# TODO: Solve the practice questions. Use separate queries and label each one.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Submission checklist

- Original Windows CSVs remain unchanged.
- Headerless copies were created before HDFS upload.
- Each dataset has a separate HDFS directory.
- Raw external and curated columnar tables are distinguishable.
- Counts, grains, null behavior, and join cardinality are documented.
- No completed solution was copied from the earlier MySQL notebooks.
